In [1]:
import torch
import random
import calendar

import platform
import sys

# Проверка доступности GPU (как мы делали раньше)
# Если CUDA есть, используем её. Если нет - процессор.

print(f"torch.version.cuda: {torch.version.cuda}")       # Должно показать '12.4' или '12.1' (не None!)
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}") # Должно быть True
print(f"torch.xpu.is_available(): {torch.xpu.is_available()}") 

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    # Проверяем атрибут xpu, чтобы код не падал на машинах со старым PyTorch
    elif hasattr(torch, 'xpu') and torch.xpu.is_available():
        return torch.device("xpu")
    else:
        return torch.device("cpu")

DEVICE = get_device()


print(f"✅ Устройство для вычислений: {DEVICE}")
# Если XPU активен, полезно вывести имя карты:
if DEVICE.type == 'xpu':
    print(f"   Карта: {torch.xpu.get_device_name(0)}")
elif DEVICE.type == 'cuda':
    print(f"   Карта: {torch.cuda.get_device_name(0)}")


torch.version.cuda: None
torch.cuda.is_available(): False
torch.xpu.is_available(): True
✅ Устройство для вычислений: xpu
   Карта: Intel(R) Graphics [0x7d45]


In [2]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    """
    Добавляет информацию о порядке элементов. Без этого трансформер 
    воспринимает 'Я ем суп' и 'Суп ем я' абсолютно одинаково.
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # register_buffer означает, что это часть состояния модели (сохраняется в .pth),
        # но не обучается (градиенты не нужны).
        self.register_buffer('pe', pe.unsqueeze(0)) # [1, MaxLen, D_model]

    def forward(self, x):
        # x: [Batch, SeqLen, D_model]
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerClassifier(nn.Module):
    def __init__(self, 
                 vocab_size, 
                 num_classes, 
                 d_model=256,      # Размерность эмбеддинга (шире, чем в RNN)
                 nhead=8,          # Количество голов внимания
                 num_layers=4,     # Глубина сети
                 dim_feedforward=1024, 
                 dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        
        # 1. Слой Эмбеддинга
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # 2. Позиционное кодирование
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # 3. Сам Трансформер (энкодер)
        # batch_first=True критически важен, чтобы вход был (Batch, Seq), а не (Seq, Batch)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout,
            batch_first=True 
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 4. Голова классификации (Classification Head)
        # Принимает усредненный вектор всего предложения
        self.classifier = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, num_classes)
        )

        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.classifier[0].bias.data.zero_()
        self.classifier[0].weight.data.uniform_(-initrange, initrange)
        self.classifier[3].bias.data.zero_()
        self.classifier[3].weight.data.uniform_(-initrange, initrange)

    def forward(self, src, padding_mask=None):
        """
        src: [Batch, Seq_Len] - индексы токенов
        padding_mask: [Batch, Seq_Len] - True там, где ПУСТОТА (padding token), False где данные
        """
        # 1. Embeddings
        # Масштабирование sqrt(d_model) — стандарт из статьи "Attention is All You Need"
        src = self.embedding(src) * math.sqrt(self.d_model)
        
        # 2. Positional Encoding
        src = self.pos_encoder(src)
        
        # 3. Transformer Pass
        # src_key_padding_mask говорит механизму Attention игнорировать нули (padding)
        output = self.transformer_encoder(src, src_key_padding_mask=padding_mask)
        # output: [Batch, Seq_Len, D_model]
        
        # 4. Pooling (Агрегация)
        # Нам нужно превратить последовательность векторов в один вектор для классификации.
        # Вариант А (CLS token): брать output[:, 0, :]
        # Вариант Б (Mean Pooling): среднее по всем словам. Более устойчиво для простых задач.
        
        # Чтобы правильно усреднить, нужно игнорировать паддинг:
        if padding_mask is not None:
            # Инвертируем маску (1 где данные, 0 где паддинг) и добавляем измерение
            mask = (~padding_mask).unsqueeze(-1).float() # [Batch, Seq, 1]
            output = output * mask # Зануляем вектора паддинга
            sum_vectors = output.sum(dim=1) # Сумма по длине
            count_vectors = mask.sum(dim=1).clamp(min=1e-9) # Количество реальных слов
            pooled_output = sum_vectors / count_vectors
        else:
            pooled_output = output.mean(dim=1)
            
        # 5. Prediction
        logits = self.classifier(pooled_output)
        return logits

In [3]:
# ПРЕДВАРИТЕЛЬНАЯ НАСТРОЙКА (один раз)
# vocab_size и num_classes возьмите из вашего датасета
model = TransformerClassifier(vocab_size=20000, num_classes=5, d_model=256).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # AdamW лучше для трансформеров
criterion = nn.CrossEntropyLoss()

# --- В ЦИКЛЕ ОБУЧЕНИЯ (batch loop) ---
def train_step(batch_data, batch_labels):
    # batch_data: [Batch, Seq_Len] (ваши индексы слов)
    # Предположим, что 0 - это padding_idx в вашем словаре
    
    # 1. Создаем маску (True там, где паддинг)
    # Это критически важно! Иначе модель будет учиться на "пустоте"
    padding_mask = (batch_data == 0) 
    
    optimizer.zero_grad()
    
    # 2. Forward
    predictions = model(batch_data, padding_mask=padding_mask)
    
    # 3. Loss & Backprop
    loss = criterion(predictions, batch_labels)
    loss.backward()
    
    # Clip gradients - полезно для стабилизации трансформеров
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    optimizer.step()
    return loss.item()

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from collections import Counter
import time
import re

# --- 1. НАСТРОЙКИ ---
BATCH_SIZE = 128   # Можно 256, если памяти много
MAX_VOCAB = 20000  # Топ-20000 слов
MAX_LEN = 100      # Обрезаем длинные новости, короткие дополняем


# --- 2. ЗАГРУЗКА ДАННЫХ (Hugging Face) ---
# Загружаем датасет (он скачается автоматически)
dataset = load_dataset("ag_news")
print(f"Пример текста: {dataset['train'][0]['text']}")
print(f"Пример лейбла: {dataset['train'][0]['label']}") # 0-3

# --- 3. ТОКЕНИЗАЦИЯ И СЛОВАРЬ ---
print("Строим словарь...")
tokenizer = lambda x: re.findall(r'\w+', x.lower()) # Простой разбиватель на слова

# Считаем частоту всех слов
word_counts = Counter()
for text in dataset['train']['text']:
    word_counts.update(tokenizer(text))

# Создаем маппинг: Слово -> Индекс
# 0: <pad>, 1: <unk> (незнакомое слово)
vocab = {"<pad>": 0, "<unk>": 1}
# Берем самые популярные слова
for word, _ in word_counts.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

print(f"Размер словаря: {len(vocab)}")

# Функция: Текст -> Список индексов
def text_pipeline(text):
    tokens = tokenizer(text)
    # Если слово есть в словаре - берем ID, иначе ID <unk>
    ids = [vocab.get(token, 1) for token in tokens]
    # Обрезка по длине
    if len(ids) > MAX_LEN:
        ids = ids[:MAX_LEN]
    return ids

# --- 4. COLLATE FUNCTION (ВАЖНО!) ---
# Эта функция вызывается DataLoader-ом, чтобы собрать пачку примеров в один тензор.
# Она делает PADDING (дополняет нулями короткие фразы).

def collate_batch(batch):
    label_list, text_list = [], []
    
    for item in batch:
        label_list.append(item['label'])
        processed_text = text_pipeline(item['text'])
        text_list.append(torch.tensor(processed_text, dtype=torch.long))
    
    # Превращаем лейблы в тензор
    label_tensor = torch.tensor(label_list, dtype=torch.long)
    
    # Pad sequence делает магию: [3, 5] и [1, 2, 3] превращает в [[3, 5, 0], [1, 2, 3]]
    text_tensor = torch.nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=0)
    
    return text_tensor, label_tensor

# Создаем DataLoader с нашей collate функцией
train_loader = DataLoader(dataset['train'], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch, num_workers=4)
test_loader = DataLoader(dataset['test'], batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=4)

# --- 5. МОДЕЛЬ (Ваш TransformerClassifier) ---
# Копируем класс TransformerClassifier сюда или убеждаемся, что он определен выше.
# Я предполагаю, что класс TransformerClassifier уже есть в памяти ноутбука из прошлого шага.

model = TransformerClassifier(
    vocab_size=len(vocab),
    num_classes=4,      # В AG News 4 класса
    d_model=256,
    nhead=8,
    num_layers=2,       # Для начала хватит 2 слоев
    dropout=0.2         # Чуть больше дропаута для реальных данных
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # Понизил LR, реальные данные капризнее
criterion = nn.CrossEntropyLoss()

# --- 6. ОБУЧЕНИЕ С МЕТРИКАМИ ---
LABELS_MAP = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

def train_epoch(epoch_index):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    start_time = time.time()
    
    for i, (text, label) in enumerate(train_loader):
        text, label = text.to(DEVICE), label.to(DEVICE)
        
        # Создаем маску паддинга: True там, где 0
        padding_mask = (text == 0)
        
        optimizer.zero_grad()
        output = model(text, padding_mask=padding_mask)
        loss = criterion(output, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        correct += (output.argmax(1) == label).sum().item()
        total += label.size(0)
        
        # Логирование каждые 100 батчей
        if i % 100 == 0 and i > 0:
            acc = 100 * correct / total
            print(f"  Batch {i} | Loss: {loss.item():.4f} | Acc: {acc:.2f}%")
            
            # --- ДЕМОНСТРАЦИЯ ПРЕДСКАЗАНИЯ ---
            # Берем первый пример из батча, декодируем и показываем
            # Обратная операция: ID -> Слово (для красоты)
            idx2word = {v: k for k, v in vocab.items()}
            sample_ids = text[0].cpu().tolist()
            # Убираем паддинги (0) для печати
            sample_text = " ".join([idx2word.get(idx, "") for idx in sample_ids if idx != 0])
            
            true_lbl = LABELS_MAP[label[0].item()]
            pred_lbl = LABELS_MAP[output[0].argmax().item()]
            print(f"  > Text: {sample_text[:100]}...") # Печатаем первые 100 символов
            print(f"  > Pred: {pred_lbl} | True: {true_lbl}\n")

    print(f"Epoch {epoch_index} Done. Time: {time.time() - start_time:.1f}s")

# --- ЗАПУСК ---
print("Start Training...")
for epoch in range(3): # 3 эпох хватит, чтобы пробить 85-90% точности
    train_epoch(epoch + 1)



Generating test split: 100%|██████████| 7600/7600 [00:00<00:00, 698499.22 examples/s]


Пример текста: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Пример лейбла: 2
Строим словарь...
Размер словаря: 20000
Start Training...
  Batch 100 | Loss: 0.8191 | Acc: 48.04%
  > Text: pistons 87 79 over rockets the nba tips off its new season as the defending champions the <unk> pist...
  > Pred: Sports | True: Sports

  Batch 200 | Loss: 0.6159 | Acc: 62.29%
  > Text: earth 39 s a real hum <unk> literally scientists have long known that the earth rings like a giant b...
  > Pred: Business | True: Sci/Tech

  Batch 300 | Loss: 0.4414 | Acc: 69.30%
  > Text: more bombs hit thai muslim south one dead more bombs hit thailand 39 s largely muslim south on satur...
  > Pred: World | True: World

  Batch 400 | Loss: 0.2761 | Acc: 73.44%
  > Text: red bull <unk> for david david coulthard is to test for the red bull racing f1 team in jerez spain t...
  > Pred: Sports | True: Sports

  Batch 500

KeyboardInterrupt: 